In [0]:
LANDING_DIR = "/Volumes/kitsune_dev/raw/landing/"
LABELS_PATH = LANDING_DIR + "SYN DoS_labels.csv.gz"

df = spark.read.csv(LABELS_PATH, header=True)
display(df)

_c0,x
1,0
2,0
3,0
4,0
5,0
6,0
7,0
8,0
9,0
10,0


In [0]:
from pyspark.sql import functions as F

# Add ingestion metadata to every row
labels_df = (
    df
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .withColumn("_ingested_at", F.current_timestamp())
)

display(labels_df)

_c0,x,_source_file,_ingested_at
1,0,dbfs:/Volumes/kitsune_dev/raw/landing/SYN%20DoS_labels.csv.gz,2026-10-22T03:10:45.762Z
2,0,dbfs:/Volumes/kitsune_dev/raw/landing/SYN%20DoS_labels.csv.gz,2026-10-22T03:10:45.762Z
3,0,dbfs:/Volumes/kitsune_dev/raw/landing/SYN%20DoS_labels.csv.gz,2026-10-22T03:10:45.762Z
4,0,dbfs:/Volumes/kitsune_dev/raw/landing/SYN%20DoS_labels.csv.gz,2026-10-22T03:10:45.762Z
5,0,dbfs:/Volumes/kitsune_dev/raw/landing/SYN%20DoS_labels.csv.gz,2026-10-22T03:10:45.762Z
6,0,dbfs:/Volumes/kitsune_dev/raw/landing/SYN%20DoS_labels.csv.gz,2026-10-22T03:10:45.762Z
7,0,dbfs:/Volumes/kitsune_dev/raw/landing/SYN%20DoS_labels.csv.gz,2026-10-22T03:10:45.762Z
8,0,dbfs:/Volumes/kitsune_dev/raw/landing/SYN%20DoS_labels.csv.gz,2026-10-22T03:10:45.762Z
9,0,dbfs:/Volumes/kitsune_dev/raw/landing/SYN%20DoS_labels.csv.gz,2026-10-22T03:10:45.762Z
10,0,dbfs:/Volumes/kitsune_dev/raw/landing/SYN%20DoS_labels.csv.gz,2026-10-22T03:10:45.762Z


In [0]:
# Write labels to bronze as a managed Delta table
(
    labels_df
    .write
    .mode("overwrite")
    .saveAsTable("kitsune_dev.bronze.syn_dos_labels_raw")
)

In [0]:
%sql
-- Table comment
COMMENT ON TABLE kitsune_dev.bronze.syn_dos_labels_raw IS 'Raw labels for the Kitsune SYN DoS capture, one row per packet in chronological order. Loaded as-is from SYN DoS_labels.csv.gz (all strings). Not cleaned.';

-- Column comments
ALTER TABLE kitsune_dev.bronze.syn_dos_labels_raw ALTER COLUMN _c0 COMMENT 'Row number from the source file (1-based), likely added when the file was exported. Used in silver to align each label with its feature row.';
ALTER TABLE kitsune_dev.bronze.syn_dos_labels_raw ALTER COLUMN x COMMENT 'Packet label: 1 = malicious (attack), 0 = benign. Renamed to label in silver.';
ALTER TABLE kitsune_dev.bronze.syn_dos_labels_raw ALTER COLUMN _source_file COMMENT 'Path of the source file this row was read from, filled by Spark (_metadata.file_path).';
ALTER TABLE kitsune_dev.bronze.syn_dos_labels_raw ALTER COLUMN _ingested_at COMMENT 'Timestamp (UTC) when the row was loaded into bronze.';

-- Add tags to the tables
ALTER TABLE kitsune_dev.bronze.syn_dos_labels_raw
SET TAGS ('layer' = 'bronze', 'source' = 'uci_kitsune', 'dataset' = 'syn_dos', 'contains_pii' = 'false');